# GrepSeek — interactive demo (Colab)

[GrepSeek](https://github.com/alirezasalemi7/grepseek) answers questions by **searching a raw Wikipedia corpus with shell commands** (`rg`/`grep`) — *Direct Corpus Interaction* — instead of a dense/sparse index. This notebook spins up the **GRPO** model with vLLM, points the repo's own agent harness at it, and lets you ask questions and watch the agent search.

> **Runtime requirements (read first).**
> - **GPU:** the model is **9B** (bf16 ≈ 18 GB). Use a Colab **A100 (40 GB)** or **L4 (24 GB)** runtime — *Runtime → Change runtime type → GPU*. The free **T4 (16 GB) is not enough**.
> - **Disk:** the full corpus is **~14 GB** on disk (≈5 GB download). Colab gives ~100 GB, so this fits — but on Colab CPU each `rg` over 14 GB takes ~10–30 s, so a query with a few tool calls takes ~1 min. If that's too slow or disk is tight, use the **optional sub-corpus** cell (Step 2b).
> - The model + dataset are public on the Hub; the **code repo must be public** for the `git clone` in Step 1 (this notebook reuses the repo's harness — it does not reimplement it).

## Step 0 — Check the GPU and disk

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU — set Runtime -> GPU (A100/L4)'
!df -h /content | tail -1


## Step 1 — Install dependencies & clone the repo

`ripgrep` (the agent's search tool), vLLM 0.17 + the Qwen3.5 transformers build (same recipe as `TRAINING_ENV.md`), and the GrepSeek repo (we **reuse** its `inference/` harness and `rl/serve_rl.sh`).

In [ ]:
# 1a. ripgrep — the shell tool the agent calls
!apt-get -qq update && apt-get -qq install -y ripgrep >/dev/null && rg --version | head -1

# 1b. serving + client stack (this pulls a self-consistent vLLM tree)
!pip -q install vllm==0.17.0 openai huggingface_hub

# 1c. pin the Qwen3.5-capable transformers build (matches the repo's verified recipe).
#     --no-deps so it doesn't refight vLLM's pins. If vLLM 0.17 already serves
#     Qwen3.5 on your runtime you can skip this; it's here for parity with the paper env.
!pip -q install --no-deps 'transformers @ git+https://github.com/huggingface/transformers.git@f048e845684894fe60440bb8506f26ffaf7b69ac'

# 1d. the GrepSeek repo (reused harness; must be a public repo)
![ -d grepseek ] || git clone --depth 1 https://github.com/alirezasalemi7/grepseek
import sys; sys.path.insert(0, '/content/grepseek')
print('repo + deps ready')


## Step 2 — Download the full Wikipedia corpus (~14 GB)

Reuses the repo's downloader (`PeterJinGo/wiki-18-corpus` → `wiki_corpus.jsonl`). ~5 GB download, decompresses to ~14 GB. This is the part to watch on Colab; if it's too slow/large, skip to **Step 2b**.

In [ ]:
!python grepseek/sft/data_generation/download_corpus.py --dest /content/data/wiki_18_corpus
CORPUS_DIR = '/content/data/wiki_18_corpus'
!ls -lh /content/data/wiki_18_corpus/wiki_corpus.jsonl


### Step 2b (optional) — sub-corpus fallback

Only if the full corpus is too slow or disk is tight. This keeps the first N passages so `rg` is fast. **Note:** answers to arbitrary questions may not be in a small slice — this is for a quick smoke, not faithful eval. Skip if Step 2 worked.

In [ ]:
# Uncomment to use a sub-corpus instead of the full one.
# N = 2_000_000   # ~ first 2M of 21M passages
# import os; os.makedirs('/content/data/wiki_18_corpus_small', exist_ok=True)
# !head -n {N} /content/data/wiki_18_corpus/wiki_corpus.jsonl > /content/data/wiki_18_corpus_small/wiki_corpus.jsonl
# CORPUS_DIR = '/content/data/wiki_18_corpus_small'
# !ls -lh {CORPUS_DIR}/wiki_corpus.jsonl


## Step 3 — Serve the GRPO model with vLLM

Reuses `rl/serve_rl.sh` (just `vllm serve` + the Qwen3 reasoning/tool-calling flags) in the background, then waits until the server answers. First run downloads the ~18 GB model, so expect a few minutes.

In [ ]:
import os, subprocess, time, urllib.request
MODEL = 'alireza7/GrepSeek-Qwen3.5-9B-GRPO'
PORT  = 8000
env = {**os.environ,
       'MODEL_PATH': MODEL, 'TP_SIZE': '1', 'PORT': str(PORT), 'HOST': '127.0.0.1',
       'GPU_UTIL': '0.90', 'MAX_NUM_SEQS': '16', 'SERVED_MODEL_NAME': 'grepseek'}
logf = open('/content/vllm_server.log', 'w')
server = subprocess.Popen(['bash', 'grepseek/rl/serve_rl.sh'], env=env,
                          stdout=logf, stderr=subprocess.STDOUT)
print(f'launched vLLM (pid={server.pid}); waiting for http://127.0.0.1:{PORT} ...')
ready = False
for i in range(120):  # up to ~20 min
    if server.poll() is not None:
        print('server exited early — tail of log:'); print(open('/content/vllm_server.log').read()[-3000:]); break
    try:
        urllib.request.urlopen(f'http://127.0.0.1:{PORT}/v1/models', timeout=5); ready = True; break
    except Exception:
        time.sleep(10);  print(f'  ...loading ({(i+1)*10}s)', end='\r')
print('\nSERVER READY' if ready else '\nnot ready — check /content/vllm_server.log')


## Step 4 — Wire the repo's agent harness to the server

We **reuse** `inference.agent.run_agent_on_example` (the exact loop used for the paper's eval): it asks the served model, parses its `<tool_call>`, runs `rg` over `CORPUS_DIR` via the repo's `tools.run_tool`, feeds back `<tool_response>`, and repeats until `<answer>`.

In [ ]:
from openai import OpenAI
from transformers import AutoTokenizer
from inference.agent import run_agent_on_example   # reused harness

client = OpenAI(base_url=f'http://127.0.0.1:{PORT}/v1', api_key='EMPTY')
tokenizer = AutoTokenizer.from_pretrained(MODEL)
print('agent ready; corpus =', CORPUS_DIR)


## Step 5 — Ask GrepSeek a question

Edit the question and re-run. `ask()` calls the reused harness and pretty-prints the full trajectory (reasoning → shell command → retrieved snippet → answer).

In [ ]:
import json

def ask(question, max_turns=6, temperature=0.6):
    rec = run_agent_on_example(
        {'id': 'q', 'question': question},
        client=client, model='grepseek', tokenizer=tokenizer,
        corpus_dir=CORPUS_DIR, max_assistant_turns=max_turns,
        temperature=temperature,
    ).to_dict()
    print('Q:', question, '\n' + '='*80)
    for m in rec['messages']:
        role = m.get('role')
        if role == 'assistant':
            print('\n[assistant]\n' + (m.get('content') or '').strip())
        elif role == 'tool':
            try:
                obj = json.loads(m.get('content') or '{}')
                out = (obj.get('stdout') or '').strip()
                print('  $ ' + (obj.get('command') or ''))
                print('  -> ' + (out[:500] + (' ...[truncated]' if len(out) > 500 else '')))
            except Exception:
                print('  [tool] ' + (m.get('content') or '')[:500])
    print('\n' + '='*80)
    print(f"ANSWER: {rec['prediction']!r}   "
          f"(turns={rec['n_assistant_turns']}, tool_calls={rec['n_tool_calls']}, "
          f"{rec['total_time_s']:.1f}s; gold={rec['gold_answers']})")
    return rec

_ = ask('Who wrote the novel on which the film Blade Runner is based?')


### Try your own
GrepSeek is strongest on multi-hop / exact-entity questions. A few to try:

In [ ]:
_ = ask('The Joggers are a four-piece band whose lead singer is the son of an American chemist who received the highest what?')
# _ = ask('What major city is the Faith Lutheran Middle School and High School located by?')
# _ = ask('YOUR QUESTION HERE')


## Notes

- **Speed:** plain `rg` over 14 GB on Colab CPU is the bottleneck (~10–30 s/call). The repo also ships a **sharded-parallel engine + search daemon** (`inference/parallel_search/`) that makes this ms/query — see [`inference/README.md`](https://github.com/alirezasalemi7/grepseek/tree/main/inference). It's overkill for a single-question demo but worth it for benchmark eval.
- **Benchmark eval (EM/F1):** use the repo's `inference/run.py --datasets ...` against the same server instead of `ask()`.
- **SFT model:** swap `MODEL = 'alireza7/GrepSeek-Qwen3.5-9B-SFT'` in Step 3 to compare the pre-RL policy.
- **Shut down the server:** `server.terminate()`.

```python
# server.terminate()
```